# Notebook 3: Feature Engineering Pipeline
**Goal:** Convert raw text & IDs into ML-ready feature vectors using Spark ML Pipelines.

**Depends on:** Notebook 2 having written `data/amazon_clean.parquet`

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, Tokenizer, StopWordsRemover, HashingTF, IDF, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.sql.functions import col
import os


In [2]:
import os, subprocess
os.environ["JAVA_HOME"] = subprocess.check_output(
    ["/usr/libexec/java_home", "-v", "17"]
).decode().strip()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('Amazon_FeatureEngineering') \
    .config('spark.driver.memory', '8g') \
    .config('spark.driver.maxResultSize', '4g') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .getOrCreate()

# Explicitly DISABLE Arrow to avoid Windows socket crash (WinError 10061)
spark.conf.set('spark.sql.execution.arrow.pyspark.enabled', 'false')

print('Spark ready | driver memory=8g | maxResultSize=4g | Arrow=disabled')
print('Spark UI: http://localhost:4040')


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/06 13:33:28 WARN Utils: Your hostname, Bavishnas-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.11 instead (on interface en0)
26/05/06 13:33:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 13:33:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/06 13:33:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/06 13:33:29 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark ready | driver memory=8g | maxResultSize=4g | Arrow=disabled
Spark UI: http://localhost:4040


In [3]:
# Load cleaned data from Notebook 2
parquet_path = 'data/amazon_clean.parquet/clean_data.parquet'
abs_path = 'file:///' + os.path.abspath(parquet_path).replace('\\', '/')
df = spark.read.parquet(abs_path)
print(f'Loaded {df.count():,} rows')
df.printSchema()


Loaded 6,284,735 rows
root
 |-- parent_asin: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- title: string (nullable = true)
 |-- text: string (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- small_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- attachment_type: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- user_review_count: long (nullable = true)
 |-- product_review_count: long (nullable = true)



## Step 1: ID Encoding (StringIndexer)

In [4]:
user_indexer = StringIndexer(inputCol='user_id', outputCol='user_id_index', handleInvalid='skip')
product_indexer = StringIndexer(inputCol='parent_asin', outputCol='product_id_index', handleInvalid='skip')
print('StringIndexers defined for user_id and parent_asin')


StringIndexers defined for user_id and parent_asin


## Step 2: NLP Pipeline (TF-IDF on review text)

In [5]:
# Use 'text' column (review body) for NLP features
tokenizer = Tokenizer(inputCol='text', outputCol='words')
remover = StopWordsRemover(inputCol='words', outputCol='filtered_words')
hashing_tf = HashingTF(inputCol='filtered_words', outputCol='raw_features', numFeatures=10000)
idf = IDF(inputCol='raw_features', outputCol='tfidf_features', minDocFreq=5)
print('NLP pipeline stages: Tokenizer -> StopWordsRemover -> HashingTF -> IDF')


NLP pipeline stages: Tokenizer -> StopWordsRemover -> HashingTF -> IDF


## Step 3: Assemble & Fit Pipeline

In [6]:
assembler = VectorAssembler(
    inputCols=['tfidf_features', 'user_id_index', 'product_id_index'],
    outputCol='features',
    handleInvalid='skip'
)

pipeline = Pipeline(stages=[user_indexer, product_indexer, tokenizer, remover, hashing_tf, idf, assembler])

print('Fitting feature pipeline...')
feature_model = pipeline.fit(df)
engineered_df = feature_model.transform(df)
print('Pipeline fit complete. Sample output:')
engineered_df.select('user_id_index', 'product_id_index', 'tfidf_features', 'features').show(5, truncate=True)


Fitting feature pipeline...


26/05/06 13:33:51 WARN DAGScheduler: Broadcasting large task binary with size 42.6 MiB
26/05/06 13:33:54 WARN DAGScheduler: Broadcasting large task binary with size 42.6 MiB
26/05/06 13:33:55 WARN DAGScheduler: Broadcasting large task binary with size 42.6 MiB
26/05/06 13:33:56 WARN DAGScheduler: Broadcasting large task binary with size 56.5 MiB
26/05/06 13:35:20 WARN DAGScheduler: Broadcasting large task binary with size 56.5 MiB
26/05/06 13:35:21 WARN DAGScheduler: Broadcasting large task binary with size 56.5 MiB


Pipeline fit complete. Sample output:


26/05/06 13:35:22 WARN DAGScheduler: Broadcasting large task binary with size 62.7 MiB


+-------------+----------------+--------------------+--------------------+
|user_id_index|product_id_index|      tfidf_features|            features|
+-------------+----------------+--------------------+--------------------+
|      90192.0|         80083.0|(10000,[136,1077,...|(10002,[136,1077,...|
|      91176.0|         80083.0|(10000,[468,528,6...|(10002,[468,528,6...|
|     379443.0|         80083.0|(10000,[597,734,7...|(10002,[597,734,7...|
|     200429.0|         80083.0|(10000,[750,6168,...|(10002,[750,6168,...|
|     448760.0|         80083.0|(10000,[393,6168]...|(10002,[393,6168,...|
+-------------+----------------+--------------------+--------------------+
only showing top 5 rows


## Step 4: Save Engineered Features

In [7]:
import os
import sys

if '.' not in sys.path:
    sys.path.append('.')

from spark_write_helper import spark_write_parquet

out_dir = 'data/engineered_features.parquet'
cols_to_save = ['user_id', 'parent_asin', 'rating', 'user_id_index', 'product_id_index']

print('Saving selected columns to Parquet via Spark...')
spark_write_parquet(engineered_df, out_dir, cols=cols_to_save)

os.makedirs('models', exist_ok=True)
feature_model.write().overwrite().save('models/feature_pipeline')
print('Pipeline model saved.')


Saving selected columns to Parquet via Spark...


26/05/06 13:35:59 WARN DAGScheduler: Broadcasting large task binary with size 93.3 MiB
26/05/06 13:37:25 WARN TaskSetManager: Stage 23 contains a task of very large size (33468 KiB). The maximum recommended task size is 1000 KiB.
26/05/06 13:37:26 WARN TaskSetManager: Stage 25 contains a task of very large size (9826 KiB). The maximum recommended task size is 1000 KiB.


Pipeline model saved.
